# 🔌 Model Context Protocol (MCP) with Strands Agents

Learn how to integrate Model Context Protocol servers with Strands Agents.

## What is MCP?

Model Context Protocol (MCP) is an open standard that enables AI applications to connect with external data sources and tools.

### Benefits:
- 🔄 Standardized tool integration
- 🌐 Connect to external services
- 🔧 Reusable tool servers
- 🚀 Easy deployment

![MCP Architecture](image/mcp.png)

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel
from strands.tools import tool

print("✅ Imports successful!")

## Understanding MCP Architecture

MCP consists of:
1. **MCP Server**: Exposes tools and resources
2. **MCP Client**: Connects to servers and uses tools
3. **Protocol**: Standardized communication format

Strands Agents can act as an MCP client to use tools from MCP servers.

## MCP Server Transport Types

The MCP Python SDK (v2) supports these transports ([MCP Python SDK docs](https://py.sdk.modelcontextprotocol.io/v2/)):

### 1. 📟 Standard I/O (stdio)
**Communication**: process stdin/stdout streams. Best for CLI tools and local development.

```python
# Server
mcp.run(transport="stdio")

# Client
from mcp import stdio_client, StdioServerParameters
client = MCPClient(lambda: stdio_client(StdioServerParameters(command="python", args=["server.py"])))
```

### 2. 🌊 Streamable HTTP (recommended for HTTP)
**Communication**: HTTP POST + streamed responses on a single `/mcp` endpoint. This is the current HTTP transport in the MCP specification and the one Strands uses when you pass a `url` to `MCPClient`.

```python
# Server
mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)

# Client (Strands builds the streamable HTTP transport for you)
client = MCPClient(url="http://localhost:8000/mcp")
```

### 3. 📡 Server-Sent Events (SSE, legacy)
Still available (`mcp.run(transport="sse")` / `mcp.client.sse.sse_client`) for compatibility with older servers, but new servers should use Streamable HTTP.

> **Migration note:** the MCP Python SDK 2.x renamed `FastMCP` to `MCPServer` (`from mcp.server.mcpserver import MCPServer`) and removed `streamablehttp_client` in favor of `streamable_http_client`. See the [v1 → v2 migration guide](https://py.sdk.modelcontextprotocol.io/v2/migration/).


## Creating MCP Servers from Custom Tools

Let's convert the custom tools from notebook 02 into MCP servers using **two different transports**!

The servers will expose:
- 🧮 **calculator**: Basic math operations
- ⏰ **get_current_time**: Current date/time
- 🎥 **analyze_video**: Video analysis with Amazon Nova 2 Lite on Bedrock

### Example 1: Streamable HTTP Transport

**Best for:** web-based integrations, remote access, production deployments

We'll create `mcp_custom_tools_server.py` with `MCPServer` using the Streamable HTTP transport:


In [ ]:
# Create the MCP server file (Streamable HTTP transport)
mcp_server_code = '''#!/usr/bin/env python3
"""
MCP Server for Custom Tools - Streamable HTTP transport
This server exposes calculator, time, and video analysis tools via Model Context Protocol.

Built with MCPServer from the MCP Python SDK 2.x.
"""
import os
from datetime import datetime
from mcp.server.mcpserver import MCPServer

# Import video reader tool
from multimodal_understanding.video_reader_local import video_reader_local

# Create MCP server (MCP Python SDK 2.x: FastMCP was renamed to MCPServer)
mcp = MCPServer("Custom Tools Server")


@mcp.tool(description="Performs basic mathematical operations (add, subtract, multiply, divide)")
def calculator(operation: str, a: float, b: float) -> str:
    """
    Calculator tool for basic math operations.

    Args:
        operation: The operation to perform (add, subtract, multiply, divide)
        a: First number
        b: Second number

    Returns:
        The result of the operation as a string
    """
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }

    result = operations.get(operation, "Invalid operation")
    return str(result)


@mcp.tool(description="Gets the current date and time in UTC timezone")
def get_current_time(timezone: str = "UTC") -> str:
    """
    Get current date and time.

    Args:
        timezone: Timezone (currently only UTC supported)

    Returns:
        Current date and time as a formatted string
    """
    now = datetime.now()
    return f"Current time ({timezone}): {now.strftime('%Y-%m-%d %H:%M:%S')}"


@mcp.tool(description="""Analyze video content using Amazon Bedrock's multimodal capabilities (Amazon Nova 2 Lite).

IMPORTANT LIMITATIONS:
- Only 1 video per request
- No audio analysis (visual only)
- Cannot identify people
- Maximum file size: ~20MB
- Supported formats: mp4, mov, avi, mkv, webm
- video_path must be an absolute path
""")
def analyze_video(
    video_path: str,
    text_prompt: str = "Describe what you see in this video",
    model_id: str = "us.amazon.nova-2-lite-v1:0",
    region: str = "us-east-1"
) -> str:
    """
    Analyze video content using AWS Bedrock.

    Args:
        video_path: Absolute path to local video file
        text_prompt: Question or instruction for analyzing the video
        model_id: Bedrock model ID to use (cross-region inference profile)
        region: AWS region for Bedrock

    Returns:
        Video analysis results as a string
    """
    result = video_reader_local(
        video_path=video_path,
        text_prompt=text_prompt,
        model_id=model_id,
        region=region
    )
    return result["content"][0]["text"]


# Run the server
if __name__ == "__main__":
    # Streamable HTTP endpoint: http://127.0.0.1:8000/mcp
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)
'''

# Write the server file
with open('mcp_custom_tools_server.py', 'w') as f:
    f.write(mcp_server_code)

print("✅ MCP server file created: mcp_custom_tools_server.py")
print("\n📝 Key MCP Server Components (Streamable HTTP Transport):")
print("="*80)
print("1. from mcp.server.mcpserver import MCPServer")
print("2. mcp = MCPServer('Custom Tools Server')")
print("3. @mcp.tool() - Decorator to define tools")
print("4. mcp.run(transport='streamable-http', host='127.0.0.1', port=8000) - Start HTTP server")
print("="*80)
print("\n🌐 Server will run at: http://127.0.0.1:8000/mcp")
print("📡 Transport: Streamable HTTP")
print("✅ Best for: Web integrations, remote access, production")


### Example 2: stdio Transport (Standard I/O)

**Transport Type:** 📟 Standard I/O (stdio)

**Best for:** Command-line tools, local development, process-to-process communication

Let's create an alternative version using stdio transport:

In [ ]:
# Create stdio version of the MCP server
stdio_server_code = '''#!/usr/bin/env python3
"""
MCP Server for Custom Tools - stdio Transport
This server uses Standard I/O for process-to-process communication.

Built with MCPServer from the MCP Python SDK 2.x.
"""
import os
from datetime import datetime
from mcp.server.mcpserver import MCPServer

# Import video reader tool
from multimodal_understanding.video_reader_local import video_reader_local

# Create MCP server (MCP Python SDK 2.x: FastMCP was renamed to MCPServer)
mcp = MCPServer("Custom Tools Server - stdio")


@mcp.tool(description="Performs basic mathematical operations (add, subtract, multiply, divide)")
def calculator(operation: str, a: float, b: float) -> str:
    """
    Calculator tool for basic math operations.

    Args:
        operation: The operation to perform (add, subtract, multiply, divide)
        a: First number
        b: Second number

    Returns:
        The result of the operation as a string
    """
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }

    result = operations.get(operation, "Invalid operation")
    return str(result)


@mcp.tool(description="Gets the current date and time in UTC timezone")
def get_current_time(timezone: str = "UTC") -> str:
    """
    Get current date and time.

    Args:
        timezone: Timezone (currently only UTC supported)

    Returns:
        Current date and time as a formatted string
    """
    now = datetime.now()
    return f"Current time ({timezone}): {now.strftime('%Y-%m-%d %H:%M:%S')}"


@mcp.tool(description="""Analyze video content using Amazon Bedrock's multimodal capabilities (Amazon Nova 2 Lite).

IMPORTANT LIMITATIONS:
- Only 1 video per request
- No audio analysis (visual only)
- Cannot identify people
- Maximum file size: ~20MB
- Supported formats: mp4, mov, avi, mkv, webm
- video_path must be an absolute path
""")
def analyze_video(
    video_path: str,
    text_prompt: str = "Describe what you see in this video",
    model_id: str = "us.amazon.nova-2-lite-v1:0",
    region: str = "us-east-1"
) -> str:
    """
    Analyze video content using AWS Bedrock.

    Args:
        video_path: Absolute path to local video file
        text_prompt: Question or instruction for analyzing the video
        model_id: Bedrock model ID to use (cross-region inference profile)
        region: AWS region for Bedrock

    Returns:
        Video analysis results as a string
    """
    result = video_reader_local(
        video_path=video_path,
        text_prompt=text_prompt,
        model_id=model_id,
        region=region
    )
    return result["content"][0]["text"]


# Run the server with stdio transport
if __name__ == "__main__":
    mcp.run(transport="stdio")  # Use stdio for command-line communication
'''

# Write the stdio server file
with open('mcp_custom_tools_server_stdio.py', 'w') as f:
    f.write(stdio_server_code)

print("✅ stdio MCP server file created: mcp_custom_tools_server_stdio.py")
print("\n📝 Key Differences (stdio Transport):")
print("="*80)
print("1. mcp.run(transport='stdio') - Use stdin/stdout")
print("2. No HTTP server - direct process communication")
print("3. Invoked as: python mcp_custom_tools_server_stdio.py")
print("="*80)
print("\n📟 Transport: Standard I/O (stdio)")
print("✅ Best for: CLI tools, local development, process-to-process")
print("\n💡 Both servers expose the same tools, just different transports!")


### Transport Comparison: Streamable HTTP vs stdio

| Feature | Streamable HTTP (mcp_custom_tools_server.py) | stdio (mcp_custom_tools_server_stdio.py) |
|---------|----------------------------------|------------------------------------------|
| **Transport** | HTTP (POST + streamed responses) | Standard I/O streams |
| **Network** | ✅ Remote access | ❌ Local only |
| **URL** | http://127.0.0.1:8000/mcp | N/A (process communication) |
| **Client** | `MCPClient(url=...)` | `stdio_client(StdioServerParameters(...))` |
| **Use Case** | Web apps, production | CLI tools, local dev |
| **Firewall** | May need configuration | No network involved |
| **Deployment** | Docker, cloud services, AgentCore Runtime | Local processes |

**For this notebook, we'll use Streamable HTTP** as it is the current HTTP transport in the MCP specification.


## Using MCP Servers with Strands Agents

Now let's connect to our **Streamable HTTP MCP server** and use it with Strands Agents.

### Step 1: Start the MCP Server

Open a **separate terminal** (same virtual environment, from the `notebook/` folder) and run:

```bash
python mcp_custom_tools_server.py
```

The server will listen on `http://127.0.0.1:8000/mcp`.

### Step 2: Connect to the MCP Server


In [ ]:
from strands.tools.mcp import MCPClient

# Connect to our custom tools MCP server via Streamable HTTP.
# Passing `url=` makes MCPClient build the streamable HTTP transport for us.
mcp_client = MCPClient(url="http://127.0.0.1:8000/mcp")

print("✅ MCP Client created (Streamable HTTP Transport)")
print("🌐 Connecting to: http://127.0.0.1:8000/mcp")
print("⚠  Make sure mcp_custom_tools_server.py is running in another terminal!")
print("\n💡 Alternative: For stdio transport, use:")
print("   from mcp import stdio_client, StdioServerParameters")
print("   client = MCPClient(lambda: stdio_client(StdioServerParameters(command='python', args=['mcp_custom_tools_server_stdio.py'])))")


### Step 3: Create Agent with MCP Tools

Strands Agents supports two approaches for MCP integration:

#### Option A: Managed Integration (Recommended - Experimental)
The agent automatically manages the MCP connection lifecycle:

In [ ]:
# Setup Bedrock model (Claude Haiku 5.5, cross-region inference profile)
session = boto3.Session(region_name='us-east-1')
bedrock_model = BedrockModel(
    model_id="global.anthropic.claude-haiku-5-5",
    boto_session=session
)

# The video tool needs absolute paths
import os
DATA = os.path.abspath("multimodal_understanding/data-sample")

# Create agent with MCP client directly (managed lifecycle)
mcp_agent = Agent(
    model=bedrock_model,
    tools=[mcp_client],  # Pass MCPClient directly!
    system_prompt=f"""You are a helpful assistant with access to:
    - Calculator for math operations
    - Time information
    - Video analysis capabilities (videos live in {DATA}; always pass absolute paths)
    """
)

print("✅ Agent created with MCP tools (managed mode)!")
print("📦 The agent will automatically connect and discover tools")


#### Option B: Manual Context Management
For production use, you can manually manage the connection:

In [ ]:
# Manual approach with explicit context management
# Uncomment to use this approach instead:

# with mcp_client:
#     # Get tools from MCP server
#     mcp_tools = mcp_client.list_tools_sync()
#     
#     # Create agent with tools
#     mcp_agent = Agent(
#         model=bedrock_model,
#         tools=mcp_tools,
#         system_prompt="You are a helpful assistant..."
#     )
#     
#     # Use agent within context
#     response = mcp_agent("What is 10 + 20?")
#     print(response)

print("ℹ️  Manual approach commented out - using managed mode above")

### Step 4: Test MCP Tools

In [ ]:
# Test 1: Calculator via MCP
print("=== 🧮 CALCULATOR TEST ===")
response = mcp_agent("What is 456 multiplied by 789?")
print(response)
print("\n" + "="*80 + "\n")

In [ ]:
# Test 2: Time via MCP
print("=== ⏰ TIME TEST ===")
response = mcp_agent("What time is it right now?")
print(response)
print("\n" + "="*80 + "\n")

In [ ]:
# Test 3: Video analysis via MCP
print("=== 🎥 VIDEO ANALYSIS TEST ===")
response = mcp_agent(
    f"Analyze the video at {DATA}/moderation-video.mp4 and describe what you see"
)
print(response)
print("\n" + "="*80 + "\n")


In [ ]:
# Test 4: Multiple tools in one request
print("=== 🔧 MULTI-TOOL TEST ===")
response = mcp_agent(
    "What time is it? Also calculate 123 + 456 for me."
)
print(response)
print("\n" + "="*80 + "\n")

## Test multimodal agents with MCP server

Let's see a practical example combining multiple MCP tools:

In [ ]:
# Real-world scenario: Video content moderation with timestamp
print("=== 🎯 PRACTICAL EXAMPLE: Content Moderation ===")

response = mcp_agent(f"""
Please help me with content moderation:
1. First, tell me what time this analysis is being done
2. Then analyze the video at {DATA}/moderation-video.mp4
3. If you see any concerning content, calculate a risk score from 0-100
""")

print(response)
print("\n" + "="*80)


## Converting Strands Tools to MCP Servers

Here's how we converted the tools from notebook 02 to an MCP server using **MCPServer** (MCP Python SDK 2.x):

### Original Strands Tool:


In [ ]:
# Original Strands tool (from notebook 02)
@tool
def calculator(operation: str, a: float, b: float) -> float:
    """Performs basic mathematical operations."""
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    return operations.get(operation, "Invalid operation")

print("✅ Original Strands tool")

### Converted to MCP Server with MCPServer:

**Option 1: Streamable HTTP Transport**
```python
# In mcp_custom_tools_server.py

from mcp.server.mcpserver import MCPServer

# Create MCP server
mcp = MCPServer("Custom Tools Server")

# Define tool with decorator
@mcp.tool(description="Performs basic mathematical operations")
def calculator(operation: str, a: float, b: float) -> str:
    """Calculator tool for basic math operations."""
    operations = {
        "add": a + b,
        "subtract": a - b,
        "multiply": a * b,
        "divide": a / b if b != 0 else "Error: Division by zero"
    }
    result = operations.get(operation, "Invalid operation")
    return str(result)

# Run with Streamable HTTP transport
if __name__ == "__main__":
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)  # 🌊 HTTP access at /mcp
```

**Option 2: stdio Transport (Process-based)**
```python
# In mcp_custom_tools_server_stdio.py

# Same tool definitions...

# Run with stdio transport
if __name__ == "__main__":
    mcp.run(transport="stdio")  # 📟 stdio transport for CLI
```

### Key Differences from Strands Tools:
1. **MCPServer Simplicity**: Much simpler than raw MCP - just use `@mcp.tool()` decorator
2. **Type Hints**: MCPServer automatically generates JSON schema from Python type hints
3. **Transport Flexibility**: choose stdio, Streamable HTTP, or SSE (legacy)
4. **Process Isolation**: Runs as separate process (HTTP server or CLI)
5. **Same Logic**: The core tool logic remains identical!
6. **Reusability**: One server, multiple clients can connect

### Transport Selection Guide:

```python
# 📟 stdio - Local CLI tools
mcp.run(transport="stdio")

# 🌊 Streamable HTTP - Web apps, production (our choice)
mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)

# 📡 SSE - legacy HTTP transport, kept for older clients
mcp.run(transport="sse", host="127.0.0.1", port=8000)
```

## Benefits of MCP Integration

### 1. Tool Reusability
Create tools once, use them across different AI applications

### 2. Standardization
Follow industry standards for tool integration

### 3. Separation of Concerns
- Tools run as separate processes
- Better security and isolation
- Independent scaling

### 4. Ecosystem
Access to growing ecosystem of MCP servers

## MCP Calculator Example

![MCP Calculator Diagram](image/mcp_calculator_diagram.png)

The diagram shows how an agent interacts with an MCP calculator server:
1. Agent receives user request
2. Agent calls MCP server tool
3. MCP server processes calculation
4. Result returned to agent
5. Agent responds to user

## Deploying MCP Servers

MCP servers can be deployed:
- 🖥️ Locally for development
- 🐳 In containers (Docker)
- ☁️ On AWS Lambda, ECS, EC2, Agentcore Runtime, AgentCore Gateway.
- 🌐 As standalone services


## Cleanup

When using managed mode, the agent handles cleanup automatically.
For manual mode, close the MCP client:

In [ ]:
# Managed mode: No cleanup needed - agent handles it automatically
print("✅ Using managed mode - automatic cleanup")

# Manual mode: Uncomment if using manual context management
# mcp_client.close()
# print("✅ MCP client closed")

## Summary

In this notebook, you learned:

✅ What Model Context Protocol (MCP) is and why it matters

✅ How to convert Strands tools to MCP servers

✅ How to connect to MCP servers from Strands Agents

✅ How to use MCP tools in your agents

✅ Real-world examples of MCP integration


### Next Steps

Continue to the next notebook to learn about State & Sessions management!